# COT Positioning — Weekly Exploration

**Purpose:** Load fresh CFTC Commitment of Traders data for all 4 markets, inspect the current positioning landscape, and generate branded charts for X/Substack content.

**Workflow:** Run all cells Sunday before market open. Review the snapshot table and charts, then write your commentary.

**Markets:**
- **NQ** — E-mini Nasdaq 100 (TFF report: Leveraged Money, Asset Managers, Dealers)
- **GC** — Gold (Disaggregated: Managed Money, Swap Dealers, Producers/Merchants)
- **CL** — WTI Crude Oil (Disaggregated)
- **ZC** — Corn (Disaggregated)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt
from datetime import date

from src.data.cot import build_cot_dataset
from src.data.config import MARKETS
from src.viz.charts import (
    chart_cot_positioning, chart_cot_overview, chart_cot_momentum
)

%matplotlib inline
pd.set_option('display.float_format', '{:,.0f}'.format)

SYMBOLS = ['NQ', 'GC', 'CL', 'ZC']
print(f"Report date: {date.today().strftime('%B %d, %Y')}")

## 1. Load COT Data

Pull processed COT data from cache. Set `force_refresh=True` on Friday evening after the CFTC release to get the latest week.

In [ ]:
FORCE_REFRESH = False  # flip to True after Friday CFTC release

cot = {}
for sym in SYMBOLS:
    cot[sym] = build_cot_dataset(sym, force_refresh=FORCE_REFRESH)
    print(f"{sym}: {len(cot[sym])} weeks | "
          f"{cot[sym].index[0].date()} → {cot[sym].index[-1].date()}")

## 2. Positioning Snapshot — Latest Week

The snapshot table gives you the current state at a glance. Key columns:
- **Net** — absolute net contracts (long - short)
- **Z-1yr / Z-3yr** — how extreme is current positioning vs recent history (>2 or <-2 = extreme)
- **26W Index** — 0-100 scale over last 26 weeks (>80 = crowded long, <20 = crowded short)
- **Pctl** — all-time percentile rank
- **WoW Chg** — week-over-week change in net contracts (momentum)

In [ ]:
GROUP_MAP = {
    'tff':          ['spec', 'comm', 'asset_mgr'],
    'disaggregated': ['spec', 'comm', 'swap'],
}
GROUP_LABELS = {
    'spec': 'Managed Money', 'comm': 'Commercials',
    'asset_mgr': 'Asset Managers', 'swap': 'Swap Dealers',
}

rows = []
for sym in SYMBOLS:
    df = cot[sym]
    latest = df.iloc[-1]
    report = MARKETS[sym]['cot_report']
    
    for group in GROUP_MAP[report]:
        net = f'{group}_net'
        rows.append({
            'Market': sym,
            'Group': GROUP_LABELS[group],
            'Net': latest[net],
            'WoW Chg': latest[f'{net}_roc'],
            'Z-1yr': latest[f'{net}_z1yr'],
            'Z-3yr': latest[f'{net}_z3yr'],
            '26W Idx': latest[f'{net}_26w'],
            'Pctl': latest[f'{net}_pctl'],
        })

snapshot = pd.DataFrame(rows)
snapshot.style.format({
    'Net': '{:,.0f}', 'WoW Chg': '{:+,.0f}',
    'Z-1yr': '{:+.2f}', 'Z-3yr': '{:+.2f}',
    '26W Idx': '{:.0f}', 'Pctl': '{:.0f}',
}).background_gradient(
    subset=['Z-1yr'], cmap='RdYlGn', vmin=-3, vmax=3
).background_gradient(
    subset=['Z-3yr'], cmap='RdYlGn', vmin=-3, vmax=3
).background_gradient(
    subset=['26W Idx'], cmap='RdYlGn', vmin=0, vmax=100
)

## 3. Open Interest Trend

Rising OI with rising price = new money entering (trend confirmation).
Falling OI with rising price = short covering rally (weaker).
Rising OI with falling price = new shorts piling in (bearish conviction).

In [ ]:
oi_rows = []
for sym in SYMBOLS:
    df = cot[sym]
    latest = df['open_interest'].iloc[-1]
    prev = df['open_interest'].iloc[-2]
    avg_26w = df['open_interest'].iloc[-26:].mean()
    chg = latest - prev
    chg_pct = (chg / prev) * 100
    vs_avg = ((latest / avg_26w) - 1) * 100
    
    oi_rows.append({
        'Market': sym,
        'OI': latest,
        'WoW Chg': chg,
        'WoW %': chg_pct,
        'vs 26W Avg %': vs_avg,
    })

oi_df = pd.DataFrame(oi_rows)
oi_df.style.format({
    'OI': '{:,.0f}', 'WoW Chg': '{:+,.0f}',
    'WoW %': '{:+.1f}%', 'vs 26W Avg %': '{:+.1f}%',
})

## 4. Extreme Detection

Flag any group where z-score exceeds +/- 2 on either the 1-year or 3-year window. These are the setups worth writing about — extreme positioning tends to mean-revert.

In [ ]:
extremes = snapshot[
    (snapshot['Z-1yr'].abs() >= 2) | (snapshot['Z-3yr'].abs() >= 2)
].copy()

if len(extremes) > 0:
    print(f"*** {len(extremes)} EXTREME POSITIONING SIGNALS ***\n")
    for _, row in extremes.iterrows():
        direction = "LONG" if row['Z-1yr'] > 0 else "SHORT"
        print(f"  {row['Market']} | {row['Group']} — extreme {direction}")
        print(f"    Z-1yr: {row['Z-1yr']:+.2f}  |  Z-3yr: {row['Z-3yr']:+.2f}  |  Pctl: {row['Pctl']:.0f}")
        print()
else:
    print("No extreme positioning signals this week. All z-scores within +/- 2.")

## 5. Biggest Weekly Moves

Who added or cut the most contracts this week? Large week-over-week changes can signal a positioning shift in progress.

In [ ]:
movers = snapshot.copy()
movers['Abs Chg'] = movers['WoW Chg'].abs()
movers = movers.sort_values('Abs Chg', ascending=False)

print("Top weekly movers (by absolute contract change):\n")
for _, row in movers.iterrows():
    direction = "added" if row['WoW Chg'] > 0 else "cut"
    print(f"  {row['Market']:>2} {row['Group']:<16} {direction} {abs(row['WoW Chg']):>8,.0f} contracts  "
          f"(net now: {row['Net']:>+10,.0f})")

---

## 6. NQ — E-mini Nasdaq 100

TFF report groups: **Leveraged Money** (spec), **Asset Managers**, **Dealers** (comm).

Leveraged Money in NQ tends to be contrarian — they're often wrong at extremes. Asset Managers are the "real money" flow and tend to be more directional.

In [ ]:
chart_cot_positioning('NQ', save=False)
plt.show()

In [ ]:
for group in ['spec', 'asset_mgr', 'comm']:
    chart_cot_overview('NQ', group=group, save=False)
    plt.show()
    chart_cot_momentum('NQ', group=group, save=False)
    plt.show()

**NQ Notes:**

_Write your observations here after reviewing the charts above._

- Managed Money positioning:
- Asset Manager flow:
- Key divergences:
- Content angle for X:

---

## 7. GC — Gold

Disaggregated report: **Managed Money** (spec), **Swap Dealers**, **Producers/Merchants** (comm).

Gold's COT is one of the most reliable contrarian indicators. When Managed Money net longs hit z-score extremes (+2 or higher), gold tends to pull back. When they get washed out (<-2), it tends to rally.

In [ ]:
chart_cot_positioning('GC', save=False)
plt.show()

In [ ]:
for group in ['spec', 'swap', 'comm']:
    chart_cot_overview('GC', group=group, save=False)
    plt.show()
    chart_cot_momentum('GC', group=group, save=False)
    plt.show()

**GC Notes:**

_Write your observations here._

- Managed Money positioning:
- Price/positioning divergence:
- Commercial hedging pressure:
- Content angle for X:

---

## 8. CL — WTI Crude Oil

Disaggregated report: **Managed Money**, **Swap Dealers**, **Producers/Merchants**.

Crude's COT has the widest swings of all 4 markets. Watch for Managed Money net positioning relative to Commercials — when specs and commercials swap extremes, it often marks turning points.

In [ ]:
chart_cot_positioning('CL', save=False)
plt.show()

In [ ]:
for group in ['spec', 'swap', 'comm']:
    chart_cot_overview('CL', group=group, save=False)
    plt.show()
    chart_cot_momentum('CL', group=group, save=False)
    plt.show()

**CL Notes:**

_Write your observations here._

- Managed Money positioning:
- Commercial hedging:
- Supply/demand context:
- Content angle for X:

---

## 9. ZC — Corn

Disaggregated report: **Managed Money**, **Swap Dealers**, **Producers/Merchants**.

Corn has the strongest seasonal patterns of the 4 markets. Commercials (grain elevators, ADM, Cargill) are the "smart money" here — when they hedge heavily, it often precedes moves. Managed Money tends to pile in late and get squeezed.

In [ ]:
chart_cot_positioning('ZC', save=False)
plt.show()

In [ ]:
for group in ['spec', 'swap', 'comm']:
    chart_cot_overview('ZC', group=group, save=False)
    plt.show()
    chart_cot_momentum('ZC', group=group, save=False)
    plt.show()

**ZC Notes:**

_Write your observations here._

- Managed Money positioning:
- Commercial hedging:
- Seasonal context (planting season approaching?):
- Content angle for X:

---

## 10. Export Charts for X/Substack

Run this cell to save all charts as branded PNGs to `output/charts/`. Pick the best ones for your thread.

In [ ]:
from src.viz.charts import chart_cot_all

for sym in SYMBOLS:
    # Main positioning chart
    chart_cot_positioning(sym, save=True)
    # All per-group charts (overview + momentum)
    chart_cot_all(sym, save=True)
    plt.close('all')

print("All COT charts exported to output/charts/")
print(f"Total: {len(SYMBOLS)} positioning + {len(SYMBOLS) * 6} per-group = {len(SYMBOLS) * 7} charts")